# MASSIVE ru-RU · квантизация и дистилляция
Режим results показывает результаты запуска. verify пересчитывает все пять метрик из исходных ответов. train повторяет эксперимент с нуля в отдельном каталоге. Реализации находятся в hw7_core.py, hw7_continue.py и awq_compat.py.

In [ ]:
import os, sys, json, subprocess, tempfile
from pathlib import Path
from pprint import pprint
MODE = 'results'
ROOT = Path.cwd()
assert (ROOT/'hw7_core.py').is_file()
ART = ROOT/'artifacts_compression_hw'
if MODE == 'train':
    run_root = Path(tempfile.mkdtemp(prefix='hw7-run-', dir=ROOT))
    print(run_root)
else:
    run_root = ROOT
env = dict(os.environ, HW7_ROOT=str(run_root))
def run(module, *args):
    subprocess.run([sys.executable, str(ROOT/module), *args], env=env, check=True)


In [ ]:
if MODE == 'train':
    for stage in ('data','baseline','teacher'):
        run('hw7_core.py', stage)
    run('hw7_core.py','teacher-answers','--samples','2048')
    from hw7_continue import QUANT, SFT
    for name in QUANT:
        run('hw7_continue.py','quant','--name',name)
    for name in SFT:
        run('hw7_continue.py','sft','--name',name)
    run('hw7_continue.py','artifacts')
elif MODE == 'verify':
    import hw7_core as core
    from hw7_continue import validate_eval
    core.OUT = ART
    for row in json.loads((ART/'results.json').read_text()):
        scores = validate_eval(row['name'])
        assert all(abs(scores[k]-row[k]) < 1e-10 for k in scores)
elif MODE != 'results':
    raise ValueError(MODE)
pprint(json.loads((run_root/'artifacts_compression_hw/results.json').read_text()))
